### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0.7)
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='**Parrots aren’t really “talking” the way humans do, but they are amazing vocal mimics.**  \nTheir ability to copy human speech (and other sounds) is the result of a mix of anatomy, brain wiring, and social needs. Here’s a breakdown of why parrots can (and often do) produce words:\n\n---\n\n## 1. The anatomy that makes mimicry possible  \n\n| Feature | What it is | Why it matters for speech‑like sounds |\n|---------|------------|--------------------------------------|\n| **Syrinx** | The bird’s vocal organ, located at the base of the trachea where it splits into the bronchi. | Unlike a mammalian larynx, the syrinx has two independent sound‑producing halves, giving parrots a huge range of pitch, tone, and rapid modulations—perfect for copying complex sounds. |\n| **Tongue & Beak** | Highly mobile, with a fine‑grained, keratinous surface. | Allows precise shaping of airflow, letting them reproduce the subtle consonant and vowel nuances of human speech. |\n| **Brain str

In [3]:
from langchain.tools import tool

@tool
def get_current_weather(location: str) -> str:
  """Get the weather at a given location"""
  return f"The weather in {location} is sunny"

model_with_tools = model.bind_tools([get_current_weather])

In [4]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
  # View tool calls made by the model
  print(f"Tool: {tool_call['name']}")
  print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks "What is the weather in New York?" Need to fetch current weather using function get_current_weather. Provide location "New York".', 'tool_calls': [{'id': 'fc_ef136154-2ae2-40e1-8a9d-cff822db4a97', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 130, 'total_tokens': 188, 'completion_time': 0.119886827, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.005790314, 'prompt_tokens_details': None, 'queue_time': 0.369895432, 'total_time': 0.125677141}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a03dd1-579d-7012-ad0d-b1b5d60dc6fe-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'fc_ef136154-2

### Tool Execution Loops

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_current_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny.


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call function get_current_weather with location "Boston".', 'tool_calls': [{'id': 'fc_b3a09be3-f7c5-4ba3-8c78-a0061f6a5c53', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_current_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 128, 'total_tokens': 170, 'completion_time': 0.088273543, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.005882699, 'prompt_tokens_details': None, 'queue_time': 0.314877708, 'total_time': 0.094156242}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_60e4b492db', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a03dd6-9075-79e1-9548-c4bea1e71f75-0', tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'Boston'}, 'id': 'fc_b3a09be